# ⚡ Análise Exploratória — Dinâmica e Perfis de Carga Horária no SIN

## Contexto

No setor elétrico, não basta saber *quanto* se consome, é preciso saber
*quando*. O perfil de carga horária é a assinatura do comportamento social
e econômico de uma região: revela quando a indústria acorda, quando o
comércio fecha, quando o calor força o ar-condicionado ao limite.

Esta análise utiliza dados horários do **ONS (Operador Nacional do Sistema
Elétrico)** de 2019 a 2026, cobrindo os quatro subsistemas do SIN: Norte,
Nordeste, Sudeste e Sul.

## Perguntas que esta EDA responde

**1. Qual é o perfil de carga diária do Brasil?**
Como o sistema se comporta hora a hora em média, onde está o vale, onde
está o pico, e qual a magnitude dessa oscilação.

**2. Qual a variabilidade ao longo do dia?**
A média esconde dispersão real. Alguns horários são previsíveis; outros
oscilam dezenas de milhares de MWmed dependendo do dia e da estação.

**3. Como o consumo muda entre dia útil e fim de semana?**
A indústria imprime uma assinatura clara na curva de carga  e quando ela
para, o sistema muda de forma.

**4. Como as regiões se comportam e como mudaram ao longo dos anos?**
Norte, Nordeste, Sudeste e Sul têm perfis completamente distintos. Algumas
regiões estão se transformando estruturalmente. A penetração solar no
Nordeste é o exemplo mais visível.

**5. Como o perfil diário varia dentro do ano?**
Verão e inverno produzem curvas diferentes. O heatmap hora × mês revela
quando cada região é mais pressionada ao longo das 24 horas.

**6. Como o consumo diário se distribui ao longo do ano?**
O calendar heatmap mostra cada dia individualmente — feriados, ondas de
calor e anomalias operacionais aparecem como exceções num padrão que,
de outra forma, pareceria uniforme.

---



## 1. Configurações

*Importações, constantes e carregamento dos dados do ONS.*

In [ ]:
import pandas as pd
import numpy as np
import requests
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from statsmodels.tsa.seasonal import MSTL
from io import BytesIO
from ipywidgets import interact, widgets
import os
from scipy.stats import gaussian_kde
import matplotlib.pyplot as plt

# ── CONSTANTES ────────────────────────────────────────────────────────
CORES_SUB  = {
    'SUDESTE': '#636EFA', 'NORDESTE': '#EF553B',
    'NORTE':   '#00CC96', 'SUL':      '#AB63FA',
    'SUDESTE/CENTRO-OESTE': '#636EFA'
}
DIAS_SEMANA_EN = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
DIAS_SEMANA_PT = ['Segunda','Terça','Quarta','Quinta','Sexta','Sábado','Domingo']
CORES_DIAS = ['#636EFA','#EF553B','#00CC96','#AB63FA','#FFA15A','#19D3F3','#FF6692']


## 1.1 Carregamento dos Dados

*Dados horários de carga do ONS (2019–2026). Salvo em Parquet para evitar redownload.*

In [ ]:
def baixar_carga_ons(anos):
    base_url = "https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/curva-carga-ho/CURVA_CARGA_{}.xlsx"
    dfs = []
    for ano in anos:
        url = base_url.format(ano)
        print(f"Baixando {ano}...")
        try:
            r = requests.get(url)
            if r.status_code == 200:
                dfs.append(pd.read_excel(BytesIO(r.content)))
                print(f"  ✅ {ano} OK")
            else:
                print(f"  ⚠️ {ano} não encontrado ({r.status_code})")
        except Exception as e:
            print(f"  ❌ {ano}: {e}")
    return pd.concat(dfs, ignore_index=True) if dfs else None

PARQUET = 'carga_diaria_ons_2019_2026.parquet'

if os.path.exists(PARQUET):
    df_carga_diaria = pd.read_parquet(PARQUET)
    print(f"✅ Dados carregados do cache ({len(df_carga_diaria):,} linhas)")
else:
    df_carga_diaria = baixar_carga_ons(list(range(2019, 2027)))
    if df_carga_diaria is not None:
        df_carga_diaria.to_parquet(PARQUET)
        print(f"💾 Salvo em {PARQUET}")

# Garantir tipos e colunas de tempo
df_carga_diaria['din_instante'] = pd.to_datetime(df_carga_diaria['din_instante'])
df_carga_diaria['hora']         = df_carga_diaria['din_instante'].dt.hour
df_carga_diaria['dia_semana']   = df_carga_diaria['din_instante'].dt.dayofweek
df_carga_diaria['mes']          = df_carga_diaria['din_instante'].dt.month
df_carga_diaria['ano']          = df_carga_diaria['din_instante'].dt.year

# Subset 2023-2026 para análises recentes
df_recente = df_carga_diaria[df_carga_diaria['din_instante'] >= '2023-01-01'].copy()

# Total nacional (soma subsistemas) — série recente
df_total = df_recente.groupby('din_instante')['val_cargaenergiahomwmed'].sum().reset_index()
df_total['hora'] = pd.to_datetime(df_total['din_instante']).dt.hour

df_carga_diaria.head()


Com os dados já estruturados em formato Parquet, evitando novos downloads e garantindo maior eficiência no carregamento, seguimos agora para a análise exploratória.

No próximo passo, vamos focar especificamente no período de 2023 a 2026, onde os padrões de carga apresentam mudanças mais recentes e relevantes para a dinâmica atual do sistema.

Vamos então visualizar esses dados da ONS nesse recorte temporal.

## 2. Carga Horária Total do SIN (2023–2026)



In [ ]:

df_recente = df_carga_diaria[df_carga_diaria['din_instante'] >= '2023-01-01'].copy()

# 2. Agrupando para o Total Nacional
df_total_recente = df_recente.groupby('din_instante')['val_cargaenergiahomwmed'].sum().reset_index()

# 3. Plotando com Plotly (Scattergl para performance)
fig = go.Figure()

fig.add_trace(go.Scattergl(
    x=df_total_recente['din_instante'],
    y=df_total_recente['val_cargaenergiahomwmed'],
    name='Total SIN (2023-2026)',
    line=dict(color='#1f77b4', width=1),
    hovertemplate='<b>Total SIN</b><br>Data: %{x}<br>Carga: %{y:.2f} MWmed<extra></extra>'
))

fig.update_layout(
    title='<b>Carga Horária Total do SIN - Foco 2023/2026</b>',
    xaxis_title='Data e Hora',
    yaxis_title='Carga (MW Médios)',
    template='plotly_white',
    hovermode='x unified',
    height=600
)

fig.update_xaxes(rangeslider_visible=True)

fig.show()

![CARGA_ONS](midia/CARGA_ONS.png)

## 🔍 Observações
A visualização da carga horária bruta (2023-2026) evidencia a magnitude do SIN, mas a alta densidade de pontos cria um ruído visual que mascara padrões e tendências. Devido a essa complexidade, seguiremos com a agregação para o nível de **carga diária**, permitindo uma análise estatística mais clara e a identificação precisa de anomalias e sazonalidades.

## 3. Perfil de Carga Médio Diário



In [ ]:
df_perfil = df_total.groupby('hora')['val_cargaenergiahomwmed'].mean().reset_index()
y_min = df_perfil['val_cargaenergiahomwmed'].min() * 0.95
y_max = df_perfil['val_cargaenergiahomwmed'].max() * 1.05

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_perfil['hora'], y=df_perfil['val_cargaenergiahomwmed'],
    mode='lines+markers', name='Média SIN',
    line=dict(color='#1A4D2E', width=4),
    marker=dict(size=8),
    hovertemplate='<b>%{x}:00</b><br>Carga: %{y:.0f} MWmed<extra></extra>'
))
fig.update_layout(
    title='<b>Perfil de Carga Média — Brasil (SIN)</b>',
    xaxis=dict(title='Hora do Dia', tickmode='linear', dtick=1),
    yaxis=dict(title='Carga Total (MWmed)', range=[y_min, y_max]),
    template='plotly_white', height=500)
fig.show()


![CARGA_DIARIA](midia/CARGA_DIARIA.png)

### Observações

O perfil médio do SIN revela uma curva com dois momentos distintos.

O **vale** ocorre entre 3h e 5h, com o sistema no limite inferior ,a indústria
em ritmo mínimo e consumo residencial próximo de zero. A partir das 6h o consumo
sobe até um primeiro platô ao redor das 11h, com leve queda no meio-dia antes
de retomar o crescimento.

O **pico** se concentra entre 18h e 19h, quando a atividade industrial ainda
opera e o consumo residencial explode com o retorno das pessoas para casa. Após
as 21h a queda é abrupta, sinalizando o encerramento do turno industrial e a
redução noturna.

A curva média, no entanto, esconde a variabilidade real do sistema. A próxima
seção sobrepõe todas as curvas diárias para revelar onde o sistema é previsível
e onde oscila.

## 4. Variabilidade do Perfil Diário


In [ ]:
# ── Preparação local ──────────────────────────────────────────────────
df_density = (df_total
              .assign(data=pd.to_datetime(df_total['din_instante']).dt.date)
              .rename(columns={'val_cargaenergiahomwmed': 'carga'})
              [['data', 'hora', 'carga']])
media = df_density.groupby('hora')['carga'].mean()

# ── Função reutilizável ───────────────────────────────────────────────
def plotar_assinatura(df_input, titulo):
    x_raw    = df_input['hora'].values
    y        = df_input['carga'].values
    x_jitter = x_raw + np.random.uniform(-0.4, 0.4, size=len(x_raw))
    xy       = np.vstack([x_jitter, y])
    z        = gaussian_kde(xy)(xy)
    idx      = z.argsort()
    x_p, y_p, z_p = x_jitter[idx], y[idx], z[idx]
    media_local = df_input.groupby('hora')['carga'].mean()

    fig = go.Figure()
    fig.add_trace(go.Scattergl(
        x=x_p, y=y_p, mode='markers',
        marker=dict(size=2.5, color=z_p, colorscale='Jet',
                    opacity=0.6, showscale=False),
        hoverinfo='skip'))
    fig.add_trace(go.Scatter(
        x=list(range(24)), y=media_local.values, mode='lines',
        line=dict(color='white', width=2.5), name='Média'))
    fig.update_layout(
        title=f'<b>{titulo}</b>',
        xaxis=dict(title='Hora do Dia', tickmode='linear', dtick=2, range=[-0.5, 23.5],
                   gridcolor='rgba(255,255,255,0.05)'),
        yaxis=dict(title='Carga (MWmed)', gridcolor='rgba(255,255,255,0.05)', zeroline=False),
        template='plotly_dark', plot_bgcolor='black', paper_bgcolor='black',
        height=600, showlegend=False)
    fig.show()

# ── Plot total ────────────────────────────────────────────────────────
plotar_assinatura(df_density, 'Assinatura Energética Contínua — SIN')

![ASSINATURA_DIARIA](midia/ASSINATURA_DIARIA.png)

### Observações

A nuvem revela o que a média escondia: o sistema é mais previsível do que
parece na madrugada e muito menos previsível durante o dia.

Entre 3h e 5h a concentração é máxima — quase todos os dias convergem para
o mesmo patamar. A dispersão cresce a partir das 6h e atinge o máximo entre
10h e 16h, quando fatores como clima, dia da semana e atividade industrial
combinam para produzir perfis completamente distintos num mesmo horário.

Os pontos isolados acima de 100k e abaixo de 50k são os eventos verdadeiramente
atípicos , ondas de calor, feriados e anomalias operacionais.

A dispersão observada sugere que nem todos os dias se comportam da mesma forma.
O próximo passo é separar esses dias por padrão de consumo, começando pelo
mais óbvio: a diferença entre dia útil e fim de semana.

## 5. Perfil por Dia da Semana



In [ ]:


fig = make_subplots(
    rows=2, cols=4,
    subplot_titles=DIAS_SEMANA_PT,
    horizontal_spacing=0.05,
    vertical_spacing=0.12)

for dia_idx, nome in enumerate(DIAS_SEMANA_PT):
    row = dia_idx // 4 + 1
    col = dia_idx % 4 + 1

    df_dia   = df_density[df_density['data'].apply(lambda x: x.weekday()) == dia_idx].copy()
    x_raw    = df_dia['hora'].values
    y        = df_dia['carga'].values
    x_jitter = x_raw + np.random.uniform(-0.4, 0.4, size=len(x_raw))
    xy       = np.vstack([x_jitter, y])
    z        = gaussian_kde(xy)(xy)
    idx      = z.argsort()
    x_p, y_p, z_p = x_jitter[idx], y[idx], z[idx]
    media_dia = df_dia.groupby('hora')['carga'].mean()

    # nuvem de pontos
    fig.add_trace(go.Scattergl(
        x=x_p, y=y_p, mode='markers',
        marker=dict(size=2, color=z_p, colorscale='Jet', opacity=0.5, showscale=False),
        hoverinfo='skip', showlegend=False),
        row=row, col=col)

    # linha de média
    fig.add_trace(go.Scatter(
        x=list(range(24)), y=media_dia.values, mode='lines',
        line=dict(color='white', width=2), showlegend=False),
        row=row, col=col)

fig.update_xaxes(tickmode='linear', dtick=6, gridcolor='rgba(255,255,255,0.05)', range=[-0.5, 23.5])
fig.update_yaxes(gridcolor='rgba(255,255,255,0.05)', zeroline=False)
fig.update_layout(
    title='<b>Assinatura Energética por Dia da Semana — SIN</b>',
    template='plotly_dark', plot_bgcolor='black', paper_bgcolor='black',
    height=600, width=1200)

fig.show()

![ASSINATURA_SEMANAL](midia/ASSINATURA_SEMANAL.png)

### Observações

Os dias úteis compartilham o mesmo perfil: consumo subindo a partir das 6h,
platô industrial durante o dia e pico noturno entre 18h e 20h. A diferença
entre segunda e sexta é pequena, a indústria segue uma rotina previsível.

O fim de semana quebra esse padrão. O platô diurno encolhe com a redução da
atividade industrial, mas o pico noturno se mantém, puxado pelo consumo
residencial. O domingo vai mais fundo ainda: sem indústria, o perfil diurno
quase desaparece, e o único destaque é o pico da noite.

O pico noturno é o único componente verdadeiramente universal , aparece todos
os dias, independente da atividade produtiva.

## 6. Evolução Histórica por Região

*Comparação do perfil diário por subsistema em cada ano (2019–2026).*

In [ ]:


df_carga_diaria['din_instante'] = pd.to_datetime(df_carga_diaria['din_instante'])
df_carga_diaria['ano']  = df_carga_diaria['din_instante'].dt.year
df_carga_diaria['hora'] = df_carga_diaria['din_instante'].dt.hour

# subsistemas disponíveis (unifica SUDESTE/CENTRO-OESTE)
df_carga_diaria['subsistema_plot'] = df_carga_diaria['nom_subsistema'].str.replace(
    'SUDESTE/CENTRO-OESTE', 'SUDESTE', regex=False)

REGIOES_DISP = sorted(df_carga_diaria['subsistema_plot'].dropna().unique())

def plotar_ridge(regiao):
    df_sub = df_carga_diaria[
        df_carga_diaria['subsistema_plot'] == regiao
    ].copy()

    df_plot = (df_sub
               .groupby(['ano', 'hora'])['val_cargaenergiahomwmed']
               .mean()
               .reset_index())

    # normalização por ano
    df_plot['carga_norm'] = (
        df_plot.groupby('ano')['val_cargaenergiahomwmed']
        .transform(lambda x: (x - x.min()) / (x.max() - x.min())))

    anos = sorted(df_plot['ano'].unique())

    sns.set_theme(style='white', rc={'axes.facecolor': (0, 0, 0, 0)})
    pal = sns.cubehelix_palette(len(anos), rot=-0.25, light=0.7)

    g = sns.FacetGrid(
        df_plot, row='ano', hue='ano',
        aspect=12, height=1, palette=pal)

    g.map(sns.lineplot, 'hora', 'carga_norm', linewidth=2)

    for ax, (ano, color) in zip(g.axes.flat, zip(anos, pal)):
        linha = df_plot[df_plot['ano'] == ano]
        ax.fill_between(linha['hora'], linha['carga_norm'],
                        alpha=0.6, color=color)
        ax.text(-1, 0.2, str(ano), fontweight='bold',
                fontsize=11, color=color)
        ax.set_xlim(0, 23)

    g.refline(y=0, linewidth=1)
    g.figure.subplots_adjust(hspace=-0.45)
    g.set_titles('')
    g.set(yticks=[], ylabel='')
    g.despine(bottom=True, left=True)
    plt.xlabel('Hora do Dia', fontsize=12)
    plt.suptitle(f'Evolução da Assinatura Energética — {regiao}',
                 fontsize=16, fontweight='bold', y=1.02)
    plt.show()

interact(
    plotar_ridge,
    regiao=widgets.Dropdown(
        options=REGIOES_DISP,
        value='NORDESTE',
        description='Região:'))

![EVOLUCAO_REGIONAL](midia/EVOLUCAO_REGIONAL.png)

### Observações

Cada região carrega uma identidade energética distinta,  e algumas estão
se transformando.

O **Nordeste** é o caso mais dramático: o vale do meio-dia cresce
progressivamente de 2019 a 2026 e a rampa noturna fica cada vez mais íngreme.
É a assinatura direta da penetração solar , mais painéis suprimindo a demanda
líquida durante o dia e pressionando o sistema quando o sol se apaga.
Esse fenômeno é conhecido como Curva do Pato: um padrão típico de sistemas com muita energia solar, em que a geração sobe forte durante o dia (achatando o consumo líquido) e depois cai rapidamente no fim da tarde. Esse “degrau” obriga outras fontes a entrarem de forma intensa para compensar a queda da solar, especialmente no início da noite.

O **Norte** tem um segundo vale ao redor das 17h, incomum e consistente ao
longo dos anos. Provavelmente reflexo das tempestades intensas do fim de tarde
amazônico, que reduzem temporariamente a atividade externa e industrial.

O **Sudeste** é a região mais estável com  o perfil quase idêntico de 2019 a 2026,
sem transformações estruturais visíveis. Economia madura, matriz consolidada.

O **Sul** segue padrão similar ao Sudeste, com amplitude ligeiramente maior
entre vale e pico, reflexo do clima mais extremo da região.

Se os anos mostram a tendência, os meses revelam o impacto do clima. Agora, cruzamos a carga com a sazonalidade para identificar como o calor e o frio moldam o consumo dentro de cada região.

## 7. Perfil Diário por Mês



In [ ]:
# ── Heatmap hora × mês por região — evolução do perfil diário ─────────
df_heatmap = df_carga_diaria.copy()
df_heatmap['subsistema_plot'] = df_heatmap['nom_subsistema'].str.replace(
    'SUDESTE/CENTRO-OESTE', 'SUDESTE', regex=False)

REGIOES_DISP = sorted(df_heatmap['subsistema_plot'].dropna().unique())
MESES_PT = ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']

def plotar_heatmap_regiao(regiao):
    df_sub = df_heatmap[df_heatmap['subsistema_plot'] == regiao].copy()

    df_plot = (df_sub
               .groupby(['mes', 'hora'])['val_cargaenergiahomwmed']
               .mean()
               .reset_index())

    vmin = df_plot['val_cargaenergiahomwmed'].min()
    vmax = df_plot['val_cargaenergiahomwmed'].max()
    df_plot['carga_norm'] = (df_plot['val_cargaenergiahomwmed'] - vmin) / (vmax - vmin)

    # Pivot — Jan no topo, Dez embaixo
    matriz = df_plot.pivot(index='mes', columns='hora', values='carga_norm')

    fig = go.Figure(go.Heatmap(
        z=matriz.values[::-1],
        x=list(range(24)),
        y=MESES_PT[::-1],
        colorscale='RdBu_r',
        zmin=0, zmax=1,
        hovertemplate='Hora: %{x}:00<br>Mês: %{y}<br>Índice: %{z:.2f}<extra></extra>',
        showscale=True,
        colorbar=dict(title='Índice<br>normalizado', thickness=15)
    ))

    fig.update_layout(
        title=f'<b>Perfil Diário por Mês — {regiao}</b><br>'
              f'<sup>Índice normalizado — vermelho = pico, azul = vale</sup>',
        xaxis=dict(title='Hora do Dia', tickmode='linear', dtick=1),
        yaxis=dict(title='Mês'),
        template='plotly_white',
        height=500, width=900)

    fig.show()

interact(
    plotar_heatmap_regiao,
    regiao=widgets.Dropdown(
        options=REGIOES_DISP,
        value='NORDESTE',
        description='Região:'))

![CARGA_SAZONAL](midia/CARGA_SAZONAL.png)

### Observações

Os heatmaps por região revelam identidades energéticas distintas em cada
subsistema.

O **Nordeste** tem o padrão mais contrastado: vale profundo no meio-dia de
jun/jul — assinatura direta da geração solar suprimindo a demanda líquida —
e pico noturno intenso o ano todo. 

O **Norte** é o mais invertido de todos. O vale se concentra na madrugada
de jan/jul, enquanto o pico domina o fim do dia em ago/set/out — a estação
seca amazônica aquecendo o sistema exatamente quando os reservatórios estão
no nível mais baixo.

O **Sudeste** tem o padrão mais claro e simétrico: vale de madrugada
universal em todos os meses, pico às 18h-19h consistente, e o inverno
(mai/ago) aprofundando o vale da manhã. É a região mais previsível do SIN.

O **Sul** é o que tem maior amplitude sazonal. O inverno (abr/ago) produz
os vales de madrugada mais intensos do sistema — o frio gaúcho reduz a
atividade a zero nas primeiras horas do dia. O verão (jan/fev) concentra
os picos mais extremos, com o calor e o ar-condicionado levando o consumo
noturno ao máximo.

## 8. Heatmap Calendário



In [ ]:
# ── Calendar Heatmap — consumo diário total do SIN ───────────────────

df_cal = df_carga_diaria.copy()
df_cal['data'] = pd.to_datetime(df_cal['din_instante']).dt.normalize()
df_cal['ano']  = df_cal['data'].dt.year

# Carga média diária total
df_diario = (df_cal
             .groupby(['data', 'ano'])['val_cargaenergiahomwmed']
             .mean()
             .reset_index())

ANOS_DISP = [2023, 2024, 2025]

def plotar_calendar(ano_sel):
    df_ano = df_diario[df_diario['ano'] == ano_sel].copy()
    df_ano = df_ano.sort_values('data').reset_index(drop=True)

    # Dia do ano (1-365)
    df_ano['dia_ano']    = df_ano['data'].dt.dayofyear
    # Dia da semana: 0=seg, 6=dom
    df_ano['dia_semana'] = df_ano['data'].dt.dayofweek
    # Semana do ano (começando em 0)
    df_ano['semana']     = (df_ano['dia_ano'] - 1 + df_ano['data'].iloc[0].weekday()) // 7

    # Normalização por ano
    vmin = df_ano['val_cargaenergiahomwmed'].min()
    vmax = df_ano['val_cargaenergiahomwmed'].max()
    df_ano['norm'] = (df_ano['val_cargaenergiahomwmed'] - vmin) / (vmax - vmin)

    fig = go.Figure(go.Heatmap(
        x=df_ano['semana'],
        y=df_ano['dia_semana'],
        z=df_ano['norm'],
        text=df_ano['data'].dt.strftime('%d/%m'),
        customdata=df_ano['val_cargaenergiahomwmed'],
        hovertemplate='<b>%{text}</b><br>Carga: %{customdata:.0f} MWmed<extra></extra>',
        colorscale='RdBu_r',
        zmin=0, zmax=1,
        showscale=True,
        colorbar=dict(title='Índice', thickness=15),
        xgap=2, ygap=2
    ))

    # Labels dos meses — posição da primeira semana de cada mês
    meses_pos = (df_ano.groupby(df_ano['data'].dt.month)['semana']
                 .min().reset_index())
    fig.update_xaxes(
        tickvals=meses_pos['semana'].tolist(),
        ticktext=MESES_PT,
        title='')

    fig.update_yaxes(
        tickvals=list(range(7)),
        ticktext=['Seg','Ter','Qua','Qui','Sex','Sáb','Dom'],
        autorange='reversed',
        title='')

    fig.update_layout(
        title=f'<b>Calendar Heatmap — Consumo Diário SIN ({ano_sel})</b><br>'
              f'<sup>Vermelho = pico, Azul = vale — escala relativa ao ano</sup>',
        template='plotly_white',
        height=320, width=1100,
        margin=dict(t=80, b=40, l=60, r=40))

    fig.show()

interact(
    plotar_calendar,
    ano_sel=widgets.Dropdown(
        options=ANOS_DISP,
        value=2023,
        description='Ano:'))

![CARGA_ANUAL](midia/CARGA_ANUAL.png)

### Observações

O calendar heatmap entrega o que nenhuma outra visualização consegue: ver
cada dia do ano individualmente.

O padrão semanal salta aos olhos imediatamente. Sábado e domingo são
consistentemente azuis ao longo de todo o ano, enquanto os dias úteis
dominam o vermelho. A indústria para, o sistema respira.

Os **feriados** aparecem como anomalias azuis isoladas em meio a semanas
vermelhas — quadrados frios em janeiro, abril e novembro que o modelo de
série temporal não consegue prever sem uma variável de calendário explícita.

O **verão** (jan-mar) concentra os picos mais intensos, puxados pelo calor
e pelo ar-condicionado. O **inverno** (jun-ago) mostra dias úteis mais
frios, mas ainda bem acima dos fins de semana.

O gráfico também revela variação dia a dia dentro da própria semana útil,
reflexo de temperatura e eventos que só aparecem nessa granularidade.

## Conclusão

Seis perguntas, seis respostas — e algumas surpresas pelo caminho.

O perfil médio do SIN tem forma clara: vale entre 3h e 5h, subida gradual
a partir das 6h e pico entre 18h e 19h, quando indústria e residencial
se somam. Mas a média esconde o que realmente importa: a dispersão entre
10h e 16h é a maior do dia, reflexo de um sistema sensível a clima,
calendário e atividade econômica.

Dia útil e fim de semana produzem dois sistemas diferentes. Sábado e
domingo apagam o platô industrial do meio do dia — o consumo diurno
despenca, mas o pico noturno persiste, sustentado pelo consumo residencial.
Feriados aparecem como anomalias azuis isoladas no calendar heatmap,
lembrando que qualquer modelo preditivo que ignore o calendário vai errar
sistematicamente nesses dias.

As quatro regiões têm identidades energéticas distintas e algumas estão
se transformando. O Nordeste é o caso mais dramático: a duck curve se
aprofunda visivelmente de 2019 a 2026, com o vale do meio-dia crescendo
a cada ano — assinatura direta da expansão solar fotovoltaica. O Norte
tem o perfil mais exótico, com pico no segundo semestre e um segundo vale
às 17h consistente ao longo dos meses. Sudeste e Sul são os mais estáveis,
com padrões previsíveis e pouca transformação estrutural no período.

O calendar heatmap fechou o ciclo mostrando o que nenhuma outra
visualização entregou: cada dia do ano individualmente, com feriados,
ondas de calor e anomalias operacionais visíveis a olho nu.

## Trabalhos Futuros

Dois caminhos naturais emergem desta análise.

**Análise de correlação temperatura × consumo.** Esta EDA sugeriu
repetidamente que o clima explica parte significativa da variabilidade
— ondas de calor aparecem nos resíduos, o verão concentra os picos mais
intensos, o Nordeste aquece no segundo semestre. O próximo passo é
quantificar essa relação diretamente, cruzando os dados horários do ONS
com séries de temperatura do INMET por subsistema. Quanto vale um grau
Celsius adicional em MWmed? A relação é linear ou tem threshold? Essas
respostas são insumo direto para o modelo preditivo.

**Modelo preditivo com XGBoost.** Com o padrão do dado bem compreendido
e a variável de temperatura incorporada, o objetivo final é construir um
modelo de previsão de carga horária. A estratégia será comparar três
abordagens em ordem crescente de complexidade: regressão linear como
baseline, SARIMA como referência de série temporal clássica, e XGBoost
com features de calendário, lags e temperatura como modelo principal.
O desempenho será avaliado por MAPE, com validação temporal via
TimeSeriesSplit para garantir que o modelo nunca veja o futuro durante
o treino.